In [2]:
# GitaWise dataset cleaning pipeline for RAG preprocessing

from __future__ import annotations

import re
import unicodedata
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# Notebook paths are resolved relative to notebooks/ when this notebook is opened there.
DATASETS_DIR = Path("../datasets")
INPUT_PATH = DATASETS_DIR / "enriched_gita.csv"
OUTPUT_PATH = DATASETS_DIR / "clean_gita.csv"

EXPECTED_COLUMNS = [
    "ID",
    "Chapter",
    "Verse",
    "Shloka",
    "Transliteration",
    "HinMeaning",
    "EngMeaning",
    "WordMeaning",
    "Interpretation",
    "Topics",
    "EmotionTags",
    "Summary",
]

TEXT_COLUMNS = [
    "Shloka",
    "Transliteration",
    "HinMeaning",
    "EngMeaning",
    "WordMeaning",
    "Interpretation",
    "Summary",
]

METADATA_COLUMNS = ["Topics", "EmotionTags"]

QUOTE_TRANSLATION = str.maketrans(
    {
        "\u2018": "'",
        "\u2019": "'",
        "\u201a": "'",
        "\u201b": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u201e": '"',
        "\u201f": '"',
    }
)


def load_csv(path: Path) -> pd.DataFrame:
    """Load a CSV with UTF-8 handling and readable errors."""
    if not path.exists():
        raise FileNotFoundError(f"Input CSV not found: {path.resolve()}")

    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="utf-8")
    except Exception as exc:
        raise RuntimeError(f"Failed to load CSV at {path.resolve()}: {exc}") from exc


def validate_schema(df: pd.DataFrame) -> None:
    """Ensure the dataset contains every required source column."""
    missing = [column for column in EXPECTED_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(
            "Input dataset is missing required columns: "
            f"{missing}. Found columns: {list(df.columns)}"
        )


def normalize_text(value: object) -> str:
    """Normalize text while preserving Sanskrit and philosophical wording."""
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = text.translate(QUOTE_TRANSLATION)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t\f\v]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


def normalize_metadata(value: object) -> str:
    """Normalize comma-separated metadata tags for retrieval filters."""
    text = normalize_text(value).lower()
    if not text:
        return ""

    # Handles plain comma strings and common serialized list strings.
    text = re.sub(r"^\s*\[|\]\s*$", "", text)
    text = text.replace('"', "").replace("'", "")

    tags: list[str] = []
    seen: set[str] = set()

    for raw_tag in text.split(","):
        tag = re.sub(r"\s+", " ", raw_tag).strip()
        if tag and tag not in seen:
            tags.append(tag)
            seen.add(tag)

    return ", ".join(tags)


def normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Apply text and metadata normalization across the dataset."""
    df = df.copy()
    df = df.fillna("")

    for column in TEXT_COLUMNS:
        df[column] = df[column].map(normalize_text)

    for column in METADATA_COLUMNS:
        df[column] = df[column].map(normalize_metadata)

    return df


def remove_invalid_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Remove rows missing essential generated fields."""
    before = len(df)
    valid_mask = (
        df["Interpretation"].astype(str).str.strip().ne("")
        & df["Summary"].astype(str).str.strip().ne("")
    )
    cleaned = df.loc[valid_mask].copy()
    return cleaned, before - len(cleaned)


def coerce_integer_columns(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Validate Chapter and Verse as integers, removing invalid rows."""
    df = df.copy()
    before = len(df)

    for column in ["Chapter", "Verse"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df = df.dropna(subset=["Chapter", "Verse"]).copy()
    df["Chapter"] = df["Chapter"].astype(int)
    df["Verse"] = df["Verse"].astype(int)

    return df, before - len(df)


def remove_duplicate_verses(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Remove duplicate verses based on canonical Chapter and Verse."""
    before = len(df)
    cleaned = df.drop_duplicates(subset=["Chapter", "Verse"], keep="first").copy()
    return cleaned, before - len(cleaned)


def build_retrieval_text(row: pd.Series) -> str:
    """Create the exact retrieval text block used for downstream embeddings."""
    return (
        f"Chapter: {row['Chapter']}\n"
        f"Verse: {row['Verse']}\n\n"
        f"Shloka:\n{row['Shloka']}\n\n"
        f"Transliteration:\n{row['Transliteration']}\n\n"
        f"English Meaning:\n{row['EngMeaning']}\n\n"
        f"Interpretation:\n{row['Interpretation']}\n\n"
        f"Topics:\n{row['Topics']}\n\n"
        f"Emotion Tags:\n{row['EmotionTags']}\n\n"
        f"Summary:\n{row['Summary']}"
    ).strip()


def add_retrieval_text(df: pd.DataFrame) -> pd.DataFrame:
    """Add the final text field that will be embedded for retrieval."""
    df = df.copy()
    df["retrieval_text"] = df.apply(build_retrieval_text, axis=1)
    return df


def remove_empty_retrieval_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Ensure every row has non-empty retrieval text."""
    before = len(df)
    valid_mask = df["retrieval_text"].astype(str).str.strip().ne("")
    cleaned = df.loc[valid_mask].copy()
    return cleaned, before - len(cleaned)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    """Save the cleaned dataset using UTF-8 with BOM for spreadsheet safety."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def clean_dataset(input_path: Path = INPUT_PATH, output_path: Path = OUTPUT_PATH) -> pd.DataFrame:
    """Run the complete cleaning pipeline."""
    df = load_csv(input_path)
    print(f"Rows loaded: {len(df)}")

    validate_schema(df)
    df = normalize_dataframe(df)

    df, missing_removed = remove_invalid_rows(df)
    df, numeric_removed = coerce_integer_columns(df)
    df, duplicates_removed = remove_duplicate_verses(df)

    df = add_retrieval_text(df)
    df, empty_retrieval_removed = remove_empty_retrieval_rows(df)

    invalid_rows_removed = missing_removed + numeric_removed + empty_retrieval_removed

    print(f"Duplicates removed: {duplicates_removed}")
    print(f"Invalid rows removed: {invalid_rows_removed}")
    print(f"Final row count: {len(df)}")

    save_csv(df, output_path)
    print(f"Cleaned dataset saved to: {output_path.resolve()}")

    return df


clean_df = clean_dataset()
display(clean_df.head())


Rows loaded: 701
Duplicates removed: 0
Invalid rows removed: 0
Final row count: 701
Cleaned dataset saved to: C:\Users\ritam\Desktop\GitaWise\datasets\clean_gita.csv


,ID,Chapter,Verse,Shloka,Transliteration,HinMeaning,EngMeaning,WordMeaning,Interpretation,Topics,EmotionTags,Summary,retrieval_text
0,BG1.1,1,1,धृतराष्ट्र उवाच |\nधर्मक्षेत्रे कुरुक्षेत्रे स...,dhṛtarāṣṭra uvāca .\ndharmakṣetre kurukṣetre s...,।।1.1।।धृतराष्ट्र ने कहा -- हे संजय ! धर्मभूमि...,1.1 Dhritarashtra said What did my people and ...,1.1 धर्मक्षेत्रे on the holy plain? कुरुक्षेत्...,The verse questions the inevitability of confl...,"dharma, conflict, perspective, impartiality, m...","anxiety, curiosity, detachment, foreboding",Questioning actions in moral conflict requires...,Chapter: 1\nVerse: 1\n\nShloka:\nधृतराष्ट्र उव...
1,BG1.2,1,2,सञ्जय उवाच |\nदृष्ट्वा तु पाण्डवानीकं व्यूढं द...,sañjaya uvāca .\ndṛṣṭvā tu pāṇḍavānīkaṃ vyūḍha...,।।1.2।।संजय ने कहा -- पाण्डव-सैन्य की व्यूह रच...,1.2. Sanjaya said Having seen the army of the ...,1.2 दृष्ट्वा having seen? तु indeed? पाण्डवानी...,Duryodhana's impulse to act on seeing the Pand...,"emotional reactivity, wise guidance, conflict ...","anxiety, defensiveness, curiosity, determination",Pause and seek wisdom before acting on perceiv...,Chapter: 1\nVerse: 2\n\nShloka:\nसञ्जय उवाच |\...
2,BG1.3,1,3,पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् |\n...,paśyaitāṃ pāṇḍuputrāṇāmācārya mahatīṃ camūm .\...,।।1.3।।हे आचार्य ! आपके बुद्धिमान शिष्य द्रुपद...,"1.3. ""Behold, O Teacher! this mighty army of t...",1.3 पश्य behold? एताम् this? पाण्डुपुत्राणाम् ...,Krishna highlights the grandeur of the Pandava...,"dharma, wisdom, leadership, surrender, perception","awe, conflicted resolve",Recognize challenges to apply wisdom and fulfi...,Chapter: 1\nVerse: 3\n\nShloka:\nपश्यैतां पाण्...
3,BG1.4,1,4,अत्र शूरा महेष्वासा भीमार्जुनसमा युधि |\nयुयुध...,atra śūrā maheṣvāsā bhīmārjunasamā yudhi .\nyu...,।।1.4।।इस सेना में महान् धनुर्धारी शूर योद्धा ...,"1.4. Here are heroes, mighty archers, eal in b...",1.4 अत्र here? शूराः heroes? महेष्वासाः mighty...,This verse highlights the collective strength ...,"collective strength, leadership, self-awarenes...","respect, determination, humility, pride",Strength emerges when diverse talents unite wi...,Chapter: 1\nVerse: 4\n\nShloka:\nअत्र शूरा महे...
4,BG1.5,1,5,धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् |\nपु...,dhṛṣṭaketuścekitānaḥ kāśirājaśca vīryavān .\np...,"।।1.5।।धृष्टकेतु, चेकितान, बलवान काशिराज, पुरु...","1.5. ""Dhrishtaketu, chekitana and the valiant ...",1.5 धृष्टकेतुः Dhrishtaketu? चेकितानः Chekitan...,The verse acknowledges diverse human strengths...,"human potential, virtue, dharma, self-realizat...","respect, contemplation, aspiration","Greatness emerges from aligned action, not mer...",Chapter: 1\nVerse: 5\n\nShloka:\nधृष्टकेतुश्चे...
